# 00c — Extract Selected CAM_FRONT Images
## Multi-Attribute Scene Classification on nuScenes

**Purpose:** Copy CAM_FRONT keyframe images for the 150 selected scenes from the v1.0-trainval blob folders to a unified working location.

### Input
- `data/processed/v1.0-trainval/selected_scenes.csv` (from `00b_scene_selection.ipynb`)
- `data/nuscenes/v1.0-trainval/v1.0-trainval/sample_data.json` (metadata)
- `data/nuscenes/v1.0-trainval/v1.0-trainval0*_blobs/samples/CAM_FRONT/*.jpg` (raw images)

### Output
- `data/nuscenes/v1.0-trainval/samples/CAM_FRONT/*.jpg` (~6,000 files, ~5-7 GB)

### Why this notebook exists
The full v1.0-trainval CAM_FRONT images are split across 10 blob folders. The nuScenes devkit and the rest of our pipeline expect images at a single `dataroot/samples/CAM_FRONT/` location. This notebook bridges that gap by copying only the files we need for our 150 selected scenes.

### Strategy
- **Copy, don't move** — keeps blob folders intact in case something goes wrong
- **Scan all 10 blobs** for each needed file (we don't trust a heuristic mapping)
- **Skip if already present** — re-running is safe (idempotent)
- **Report missing files** — if any are not found, we know exactly which

### Expected runtime
- ~6,000 file copies
- Average file size ~1 MB
- Total data ~5-7 GB
- Time: 20-40 min depending on disk speed


## 0. Setup

In [1]:
import os
import json
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
from tqdm.auto import tqdm

print('Imports OK')

Imports OK


## 0.1 Locate Project Root

In [2]:
def find_project_root():
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError(
        f'Could not find project root from {p}. '
        f'Looking for parent containing both README.md and notebooks/'
    )

PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

Project root: C:\Users\leemi\Documents\GitHub\nuscenes-scene-classification-ml


## 1. Configuration

In [3]:
DATASET_VERSION = 'v1.0-trainval'

# Paths
NUSCENES_BASE = Path('data/nuscenes') / DATASET_VERSION
PROCESSED_DIR = Path('data/processed') / DATASET_VERSION

# Metadata location (auto-detect)
def find_metadata_dir(base, version):
    candidates = [base / version, base / f'{version}_meta' / version, base]
    for c in candidates:
        if c.exists() and len(list(c.glob('*.json'))) >= 10:
            return c
    raise FileNotFoundError(f'Could not find metadata under {base}')

METADATA_DIR = find_metadata_dir(NUSCENES_BASE, DATASET_VERSION)
print(f'Metadata dir: {METADATA_DIR}')

# Blob folders to scan
BLOB_FOLDERS = sorted(NUSCENES_BASE.glob('v1.0-trainval*_blobs'))
print(f'Blob folders found: {len(BLOB_FOLDERS)}')
for b in BLOB_FOLDERS:
    print(f'  - {b.name}')

# Destination — standard nuScenes layout
DEST_DIR = NUSCENES_BASE / 'samples' / 'CAM_FRONT'
DEST_DIR.mkdir(parents=True, exist_ok=True)
print(f'\nDestination: {DEST_DIR}')

Metadata dir: data\nuscenes\v1.0-trainval\v1.0-trainval_meta\v1.0-trainval
Blob folders found: 10
  - v1.0-trainval01_blobs
  - v1.0-trainval02_blobs
  - v1.0-trainval03_blobs
  - v1.0-trainval04_blobs
  - v1.0-trainval05_blobs
  - v1.0-trainval06_blobs
  - v1.0-trainval07_blobs
  - v1.0-trainval08_blobs
  - v1.0-trainval09_blobs
  - v1.0-trainval10_blobs

Destination: data\nuscenes\v1.0-trainval\samples\CAM_FRONT


## 2. Load Selected Scenes

In [4]:
selected_path = PROCESSED_DIR / 'selected_scenes.csv'
if not selected_path.exists():
    raise FileNotFoundError(
        f'{selected_path} does not exist.\n'
        f'Run 00b_scene_selection.ipynb first.'
    )

df_selected = pd.read_csv(selected_path)
print(f'Loaded {len(df_selected)} selected scenes')
selected_scene_tokens = set(df_selected['scene_token'].tolist())
print(f'Selected scene tokens (set): {len(selected_scene_tokens)}')

Loaded 150 selected scenes
Selected scene tokens (set): 150


## 3. Build Sample → Scene Mapping

In [5]:
# Load sample.json to map sample_token → scene_token
sample_json_path = METADATA_DIR / 'sample.json'
with open(sample_json_path) as f:
    sample_list = json.load(f)

sample_to_scene = {s['token']: s['scene_token'] for s in sample_list}
print(f'Built sample → scene mapping ({len(sample_to_scene)} samples)')

Built sample → scene mapping (34149 samples)


## 4. Identify Needed Files

Find all CAM_FRONT keyframe filenames belonging to our 150 selected scenes.

In [6]:
# Load sample_data.json (large file — ~500MB)
print('Loading sample_data.json (this may take 30-60 seconds)...')
sample_data_path = METADATA_DIR / 'sample_data.json'
with open(sample_data_path) as f:
    sample_data_list = json.load(f)
print(f'✓ Loaded {len(sample_data_list)} sample_data records')

# Filter to CAM_FRONT keyframes belonging to our selected scenes
needed = []
for sd in sample_data_list:
    if not sd.get('is_key_frame', False):
        continue
    fn = sd.get('filename', '')
    # Match CAM_FRONT specifically (not CAM_FRONT_LEFT or CAM_FRONT_RIGHT)
    if '/CAM_FRONT/' not in fn:
        continue
    sample_token = sd['sample_token']
    scene_token = sample_to_scene.get(sample_token)
    if scene_token in selected_scene_tokens:
        needed.append({
            'filename':     fn,           # e.g., samples/CAM_FRONT/xxx.jpg
            'basename':     Path(fn).name,
            'sample_token': sample_token,
            'scene_token':  scene_token,
        })

df_needed = pd.DataFrame(needed)
print(f'\n✓ Need {len(df_needed)} CAM_FRONT keyframes for 150 selected scenes')
print(f'Average per scene: {len(df_needed) / len(df_selected):.1f} keyframes')

Loading sample_data.json (this may take 30-60 seconds)...
✓ Loaded 2631083 sample_data records

✓ Need 6021 CAM_FRONT keyframes for 150 selected scenes
Average per scene: 40.1 keyframes


## 5. Index Available Files Across All Blobs

Build a lookup of `basename → blob_path` for every CAM_FRONT file in the 10 blob folders.

In [7]:
print('Indexing CAM_FRONT files across all blob folders...')

# Index: basename → full path
file_index = {}
duplicates = 0
for blob_dir in tqdm(BLOB_FOLDERS, desc='Scanning blobs'):
    cam_front_dir = blob_dir / 'samples' / 'CAM_FRONT'
    if not cam_front_dir.exists():
        print(f'  ⚠️ {blob_dir.name}: no samples/CAM_FRONT/ subfolder')
        continue
    for jpg in cam_front_dir.glob('*.jpg'):
        if jpg.name in file_index:
            duplicates += 1
        file_index[jpg.name] = jpg

print(f'\n✓ Indexed {len(file_index)} unique CAM_FRONT files')
if duplicates > 0:
    print(f'⚠️ {duplicates} duplicate filenames across blobs (last occurrence kept)')

Indexing CAM_FRONT files across all blob folders...


Scanning blobs:   0%|          | 0/10 [00:00<?, ?it/s]


✓ Indexed 34149 unique CAM_FRONT files


## 6. Copy Files

For each needed file, look it up in the index and copy to the destination. Skip files already in destination (idempotent).

In [8]:
copied = 0
skipped = 0
missing = []
total_bytes = 0

for _, row in tqdm(df_needed.iterrows(), total=len(df_needed), desc='Copying'):
    basename = row['basename']
    dest_path = DEST_DIR / basename

    if dest_path.exists():
        skipped += 1
        total_bytes += dest_path.stat().st_size
        continue

    src_path = file_index.get(basename)
    if src_path is None:
        missing.append(basename)
        continue

    shutil.copy2(src_path, dest_path)
    copied += 1
    total_bytes += dest_path.stat().st_size

print(f'\n=== COPY SUMMARY ===')
print(f'Needed:        {len(df_needed)} files')
print(f'Copied:        {copied}')
print(f'Already there: {skipped}')
print(f'Missing:       {len(missing)}')
print(f'Total size:    {total_bytes / 1e9:.2f} GB')

Copying:   0%|          | 0/6021 [00:00<?, ?it/s]


=== COPY SUMMARY ===
Needed:        6021 files
Copied:        6021
Already there: 0
Missing:       0
Total size:    0.96 GB


## 7. Missing-Files Report

In [9]:
if missing:
    print(f'⚠️ {len(missing)} files could not be found in any blob folder')
    print(f'\nFirst 10 missing:')
    for m in missing[:10]:
        print(f'  - {m}')

    # Save full missing list
    missing_path = PROCESSED_DIR / 'missing_files.txt'
    with open(missing_path, 'w') as f:
        for m in missing:
            f.write(m + '\n')
    print(f'\nFull list saved to: {missing_path}')

    # Which scenes are affected
    affected_scenes = df_needed[df_needed['basename'].isin(missing)]['scene_token'].unique()
    print(f'\nAffected scenes: {len(affected_scenes)} of {len(selected_scene_tokens)}')
else:
    print('✓ All needed files were successfully copied or already present')

✓ All needed files were successfully copied or already present


## 8. Verification

Confirm the destination has exactly the files we expect.

In [10]:
dest_files = set(p.name for p in DEST_DIR.glob('*.jpg'))
needed_files = set(df_needed['basename'].tolist())

present = needed_files & dest_files
unexpected = dest_files - needed_files
absent = needed_files - dest_files

print(f'Files needed:         {len(needed_files)}')
print(f'Files in destination: {len(dest_files)}')
print(f'  - matching:         {len(present)}')
print(f'  - unexpected extra: {len(unexpected)}')
print(f'  - still missing:    {len(absent)}')

if len(present) == len(needed_files) and len(absent) == 0:
    print('\n✓ EXTRACTION COMPLETE — all needed files present')
else:
    print(f'\n⚠️ Extraction incomplete — {len(absent)} files still missing')

Files needed:         6021
Files in destination: 6021
  - matching:         6021
  - unexpected extra: 0
  - still missing:    0

✓ EXTRACTION COMPLETE — all needed files present


## 9. Save Extraction Manifest

In [12]:
# Save the per-file manifest for downstream notebooks
df_needed['present_in_dest'] = df_needed['basename'].apply(lambda b: (DEST_DIR / b).exists())
manifest_path = PROCESSED_DIR / 'extracted_images_manifest.csv'
df_needed.to_csv(manifest_path, index=False)
print(f'Saved → {manifest_path}')

# Summary JSON
summary = {
    'dataset_version':      DATASET_VERSION,
    'n_selected_scenes':    int(len(df_selected)),
    'n_files_needed':       int(len(df_needed)),
    'n_files_present':      int(len(present)),
    'n_files_missing':      int(len(absent)),
    'total_size_gb':        round(total_bytes / 1e9, 2),
    'destination':          str(DEST_DIR),
    'extraction_complete':  bool(len(absent) == 0),
}
summary_path = PROCESSED_DIR / 'extraction_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved → {summary_path}')
print()
print(json.dumps(summary, indent=2))

Saved → data\processed\v1.0-trainval\extracted_images_manifest.csv
Saved → data\processed\v1.0-trainval\extraction_summary.json

{
  "dataset_version": "v1.0-trainval",
  "n_selected_scenes": 150,
  "n_files_needed": 6021,
  "n_files_present": 6021,
  "n_files_missing": 0,
  "total_size_gb": 0.96,
  "destination": "data\\nuscenes\\v1.0-trainval\\samples\\CAM_FRONT",
  "extraction_complete": true
}


## 10. Next Steps

In [13]:
print('=' * 70)
print('EXTRACTION COMPLETE')
print('=' * 70)
print()
print(f'Selected images now at: {DEST_DIR}')
print(f'Files: {len(present)} / {len(needed_files)}')
print(f'Size:  {total_bytes / 1e9:.2f} GB')
print()
print('Next notebook: 01_eda.ipynb (EDA on the actual selected images)')
print('Or proceed directly to 02_attribute_labels.ipynb if EDA was covered by 00a')

EXTRACTION COMPLETE

Selected images now at: data\nuscenes\v1.0-trainval\samples\CAM_FRONT
Files: 6021 / 6021
Size:  0.96 GB

Next notebook: 01_eda.ipynb (EDA on the actual selected images)
Or proceed directly to 02_attribute_labels.ipynb if EDA was covered by 00a


---
## Optional cleanup (after Stage 2 pipeline is fully working)

Once you have validated Stage 2 results, you can free disk space by deleting the blob folders:

```powershell
# DO NOT RUN until Stage 2 is complete and you're sure extraction worked
Remove-Item data/nuscenes/v1.0-trainval/v1.0-trainval0*_blobs -Recurse -Force
```

This frees ~388 GB. The extracted CAM_FRONT samples (~5-7 GB) remain in place.

Don't do this on the same day as extraction — let the pipeline run first to confirm everything works.
